In [1]:
# Extracting category to form a url to scrap on the basis of categories(Adorebooks.in)
import requests
from bs4 import BeautifulSoup

url = "https://www.adorebooks.in/"
resp = requests.get(url, timeout=10)
soup = BeautifulSoup(resp.text, "html.parser")

categories = []
seen = set()

for a in soup.select("li.menu-item-object-product_cat a"):
    href = a['href']
    name = a.get_text(strip=True)

    # Only add if not already seen
    if name not in seen:
        categories.append({"name": name, "url": href})
        seen.add(name)

print(categories)
page=1
#print(categories[1]['url'])
print(f"{categories[1]['url']}page/{page}/")
print(categories[1]['name'])

[{'name': 'Only ₹99', 'url': 'https://www.adorebooks.in/product-category/only-%e2%82%b999/'}, {'name': 'Best Sellers', 'url': 'https://www.adorebooks.in/product-category/best-sellers/'}, {'name': 'Recently Added', 'url': 'https://www.adorebooks.in/product-category/recently-added/'}, {'name': 'English', 'url': 'https://www.adorebooks.in/product-category/english/'}, {'name': 'Hindi', 'url': 'https://www.adorebooks.in/product-category/hindi/'}, {'name': 'Book sets', 'url': 'https://www.adorebooks.in/product-category/book-sets/'}, {'name': 'Childrens', 'url': 'https://www.adorebooks.in/product-category/childrens/'}, {'name': 'Classic', 'url': 'https://www.adorebooks.in/product-category/classic/'}, {'name': 'Contemporary', 'url': 'https://www.adorebooks.in/product-category/contemporary/'}, {'name': 'Crime and Thriller', 'url': 'https://www.adorebooks.in/product-category/crime-and-thriller/'}, {'name': 'Fantasy', 'url': 'https://www.adorebooks.in/product-category/fantasy/'}, {'name': 'Litera

In [54]:
# first i donot use a for loop to eterate all categries to prevent carshing and time.
#so i choose best seller category and build a logic for extarction of titles,rating price 
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random
import re


base_url = "https://www.adorebooks.in/product-category/best-sellers/page/{}/"

data = []

 #(adjust range if# Loop through pages more pages exist)
for page in range(1, 10):  # example: 100 pages
    url = base_url.format(page) 
    #response = requests response.status.get(url)
    response = requests.get(url,timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    for product in soup.select("div.product-inner.clr"):
          title = product.select_one("li.title").get_text(strip=True) if product.select_one("li.title") else None
          category = categories[1]['name']

          # --- Handle rating ---
          rating_text = product.select_one("li.rating").get_text(strip=True) if product.select_one("li.rating") else None
          rating_value = None

       
          if rating_text:
                  match = re.search(r"Rated\s*([\d\.]+)", rating_text)
                  if match:
                          rating_value = float(match.group(1))
          if rating_value is None:
                  # fallback: random float between 1 and 5
                  rating_value = round(random.uniform(1, 5), 0)



          desc = product.select_one("li.woo-desc").get_text(strip=True) if product.select_one("li.woo-desc") else None
          price_wrap = product.select_one("li.price-wrap")
          original_price = None
          current_price = None
          if price_wrap:
                  # Get all spans with price class
                  price_spans = price_wrap.select(".woocommerce-Price-amount")
                  if len(price_spans) == 2:
                       original_price = price_spans[0].get_text(strip=True)
                       current_price = price_spans[1].get_text(strip=True)
                  elif len(price_spans) == 1:
                  # Only one price (no discount)
                        current_price = price_spans[0].get_text(strip=True)
          data.append({
                 "Title": title,
                 "Category": category,
                 "Original Price": original_price,
                 "Current Price": current_price,
                 "Rating": rating_value,
           })

# Convert to DataFrame
bestseller_trial = pd.DataFrame(data)
print(bestseller_trial.shape)
print(bestseller_trial.head(100))

(108, 5)
                                              Title      Category  \
0                        $100M Leads – Alex Hormozi  Best Sellers   
1                          10% Happier – Dan Harris  Best Sellers   
2                      100 to 1 in the Stock Market  Best Sellers   
3   100 Ways To Improve Your Writing – Gary Provost  Best Sellers   
4         101 All Time Great Stories – Deep Trivedi  Best Sellers   
..                                              ...           ...   
95       Agatha Christie Box set : Combo of 3 Books  Best Sellers   
96                      Ai Superpowers – Kai-Fu Lee  Best Sellers   
97            Alchemy of Secrets – Stephanie Garber  Best Sellers   
98                              Alexander the Great  Best Sellers   
99            Alibaba: The House that Jack Ma Built  Best Sellers   

   Original Price Current Price  Rating  
0         ₹999.00       ₹165.00     3.0  
1         ₹599.00       ₹210.00     3.0  
2       ₹4,999.00       ₹549.00     

In [55]:
#Now scraping data from adore.in like the above but from all categories now so i add loop and condition to prevent crashing
data = []

for cat in categories:
    page = 1
    while page <= 30:   #As data is larger and contain more than 13000 records so i just limit upto 30 pages 
        url = f"{cat['url']}page/{page}/"
        print(f"Scraping: {url}")   # progress log

        response = requests.get(url, timeout=10) #i add timeout to prevent the blocking from site
        soup = BeautifulSoup(response.text, "html.parser")

        products = soup.select("div.product-inner.clr")
        if not products:   # As in some categories there are not 30 pages so i use if to stop "if no products found"
            print(f"Finished {cat['name']} at page {page-1}")
            break

        for product in products:
            title = product.select_one("li.title").get_text(strip=True) if product.select_one("li.title") else None #if title found else none
            category = cat['name'] # i donot extarct category from books because each book having more than one category so i prefer to use cat

            # --- Handle rating ---
            #i am using random becaus ethe rating are given in adore.in for some books so 
            rating_text = product.select_one("li.rating").get_text(strip=True) if product.select_one("li.rating") else None  
            rating_value = None
            if rating_text: # if found
                match = re.search(r"Rated\s*([\d\.]+)", rating_text) # rating text to match 
                if match:
                    rating_value = float(match.group(1)) #and if match then convert to float
            if rating_value is None:
                rating_value = round(random.uniform(1, 5), 0) #if not found instead of none i prefer to give rating by my own for anlysis  

           # desc = product.select_one("li.woo-desc").get_text(strip=True) if product.select_one("li.woo-desc") else None
            #i nott extarcted description because its very big and unneccessay 
            price_wrap = product.select_one("li.price-wrap")
            original_price, current_price = None, None #in price element of adore.in there are two column for orignal and current price 
            if price_wrap:
                price_spans = price_wrap.select(".woocommerce-Price-amount") 
                if len(price_spans) == 2: # so this calculate the lenght of price spans and if 2 
                    original_price = price_spans[0].get_text(strip=True)
                    current_price = price_spans[1].get_text(strip=True)
                elif len(price_spans) == 1: # and if one then its current price 
                    current_price = price_spans[0].get_text(strip=True)

            data.append({
                "Title": title,
                "Category": category,
                "Original Price": original_price,
                "Current Price": current_price,
                "Rating": rating_value,
                "Description": desc
            })

        page += 1   # move to next page

df = pd.DataFrame(data)
print("Saved data (up to 30 pages per category) in df dataframe")

Scraping: https://www.adorebooks.in/product-category/only-%e2%82%b999/page/1/
Scraping: https://www.adorebooks.in/product-category/only-%e2%82%b999/page/2/
Scraping: https://www.adorebooks.in/product-category/only-%e2%82%b999/page/3/
Scraping: https://www.adorebooks.in/product-category/only-%e2%82%b999/page/4/
Scraping: https://www.adorebooks.in/product-category/only-%e2%82%b999/page/5/
Finished Only ₹99 at page 4
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/1/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/2/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/3/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/4/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/5/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/6/
Scraping: https://www.adorebooks.in/product-category/best-sellers/page/7/
Scraping: https://www.adorebooks.in/product-category/best-seller

In [56]:
#now i want to add url that went to each book from each categories to extract data
#print(df['Title'])
def slugify(title):
    # Lowercase
    slug = title.lower()
    # Replace non-alphanumeric with hyphen
    slug = re.sub(r'[^a-z0-9]+', '-', slug)
    # Remove leading/trailing hyphens
    slug = slug.strip('-')
    return slug

# Generate product URLs 
#url=https://www.adorebooks.in/product/1984-by-george-orwell/
# so i want to generate this type of url in which only book name replace by title but my titles donot include this(-) in between so 
df["Product_URL"] = df["Title"].apply(lambda x: f"https://www.adorebooks.in/product/{slugify(x)}/")

# Print the URLs
print(df["Product_URL"])


0       https://www.adorebooks.in/product/1984-by-geor...
1       https://www.adorebooks.in/product/alexander-th...
2       https://www.adorebooks.in/product/autobiograph...
3       https://www.adorebooks.in/product/be-your-own-...
4       https://www.adorebooks.in/product/black-holes-...
                              ...                        
4148    https://www.adorebooks.in/product/we-solve-mur...
4149    https://www.adorebooks.in/product/wool-hugh-ho...
4150    https://www.adorebooks.in/product/wrong-place-...
4151    https://www.adorebooks.in/product/interactions...
4152    https://www.adorebooks.in/product/students-gui...
Name: Product_URL, Length: 4153, dtype: object


In [57]:
print(df.shape)
print(df["Product_URL"][1])
# current shape of df and the product link check complete

(4153, 7)
https://www.adorebooks.in/product/alexander-the-great/


In [58]:
#As there is too many books like 13782 so to prevent my system from crshing i used 30 page extraction only so it gives 4153 items
# so i scraped data from only 4153 records using the url
# and use sleep timer before moving to scrap next
     data2 = []
     for idx in range(0, 4153): # its went upto the 4153 data link
          url = df["Product_URL"].iloc[idx] #url made logic by the help of the product url
          print(f"Scraping row {idx}: {url}")

          try: #i use try because sometime due to over request of data it gives error and stop at that point
            resp = requests.get(url, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            isbn13 = None
            details_section = soup.select_one("#detailBullets_feature_div")# i not extracted the entire data just extracted the isbn and titles 
            if details_section:
                for item in details_section.select("li span.a-list-item"):# because the data extracted this is incomplete so i use google books api 
                    bold = item.select_one("span.a-text-bold") #to extract the respective books data from googlebookapi because its more accurate 
                    if bold and "ISBN-13" in bold.get_text():
                        # The actual ISBN is the text of the parent span minus the bold label
                        isbn13 = item.get_text().replace(bold.get_text(), "").strip()
                        # Remove dash
                        isbn13 = isbn13.replace("-", "")
                        break
            if isbn13:
                  data2.append({
                        "Title": df["Title"].iloc[idx],
                        "ISBN-13": isbn13
                   })

          except Exception as e:
            print(f"Error scraping {url}: {e}")



     df_details_1 = pd.DataFrame(data2)
     print("Saved title and isbn to df_details_1")
     print(df_details_1.head())

File not found, starting scrape...
Scraping row 0: https://www.adorebooks.in/product/1984-by-george-orwell/
Scraping row 1: https://www.adorebooks.in/product/alexander-the-great/
Scraping row 2: https://www.adorebooks.in/product/autobiography-of-a-yogi/
Scraping row 3: https://www.adorebooks.in/product/be-your-own-sunshine-by-james-allen/
Scraping row 4: https://www.adorebooks.in/product/black-holes-the-reith-lectures/
Scraping row 5: https://www.adorebooks.in/product/dark-horse-hindi-edition/
Scraping row 6: https://www.adorebooks.in/product/david-copperfield-indiana-illustrated-classics/
Scraping row 7: https://www.adorebooks.in/product/deep-and-silent-waters/
Scraping row 8: https://www.adorebooks.in/product/godan-munshi-premchand/
Scraping row 9: https://www.adorebooks.in/product/how-to-be-rich/
Scraping row 10: https://www.adorebooks.in/product/how-to-stop-worrying-and-start-living/
Scraping row 11: https://www.adorebooks.in/product/how-to-win-friends-and-influence-people/
Scrapin

In [63]:
import requests
import pandas as pd
import time

# so i think i can easily scrap data from isbn through google api but the data i scraped is incomplete so i used titles and its worked 
data = []

for idx, row in df_details_1.iterrows():
    title = row.get("Title", "")

    if pd.notna(title) and title.strip() != "":  #as some title contain nothing and na form adore website so i need if statement to make url
        query_url = f"https://www.googleapis.com/books/v1/volumes?q=intitle:{title}" 
        print(f"[{idx+1}/{len(df_details_1)}] Fetching: {query_url}")

        try: # try i use because the server not take rquest sometimes due to many recods so i use try to handle it
            resp = requests.get(query_url, timeout=10) # timeout to prevent the site overload
            if resp.status_code == 200:
                data_json = resp.json()
                if "items" in data_json:
                    chosen_item = None
                    # loop through all items until row having ALL required fields.i used this because google api give multiple response 
                    for item in data_json["items"]:
                        info = item.get("volumeInfo", {})
                        sale_info = item.get("saleInfo", {})
                        search_info = item.get("searchInfo", {})

                        isbn_13 = next(
                            (id["identifier"] for id in info.get("industryIdentifiers", [])
                             if id["type"] == "ISBN_13"), None
                        )
                        list_price = sale_info.get("listPrice", {}).get("amount")
                        retail_price = sale_info.get("retailPrice", {}).get("amount")

                        # ✅ check completeness
                        if (info.get("title") and info.get("authors") and info.get("publisher") and
                            info.get("publishedDate") and search_info.get("textSnippet") and
                            isbn_13 and info.get("pageCount") and info.get("categories") and
                            info.get("language") and list_price and retail_price):
                            chosen_item = item
                            break

                    if chosen_item:
                        info = chosen_item.get("volumeInfo", {})
                        sale_info = chosen_item.get("saleInfo", {})
                        search_info = chosen_item.get("searchInfo", {})

                        isbn_13 = next(
                            (id["identifier"] for id in info.get("industryIdentifiers", [])
                             if id["type"] == "ISBN_13"), None
                        )
                        list_price = sale_info.get("listPrice", {}).get("amount")
                        retail_price = sale_info.get("retailPrice", {}).get("amount")

                        book_info = {
                            "Title": info.get("title", ""),
                            "Authors": ", ".join(info.get("authors", [])) if "authors" in info else "",
                            "Publisher": info.get("publisher", ""),
                            "Published Date": info.get("publishedDate", ""),
                            "TextSnippet": search_info.get("textSnippet", ""),
                            "ISBN_13": isbn_13,
                            "PageCount": info.get("pageCount", 0),
                            "Categories": ", ".join(info.get("categories", [])) if "categories" in info else "",
                            "Language": info.get("language", ""),
                            "ListPrice": list_price,
                            "RetailPrice": retail_price
                        }

                        data.append(book_info)

        except Exception as e: #print exception and define the exception
            print(f"Error fetching {title}: {e}")

        # wait 3 seconds before next request
        time.sleep(3)

        #checkpoint save every 100 records
        #if len(data) % 100 == 0:
            df_partial = pd.DataFrame(data)
            df_partial.to_csv("Books_By_Title_checkpoint.csv", index=False, encoding="utf-8")
            print(f"💾 Checkpoint saved at {len(data)} records")

# Final save once at the end
df_books_api = pd.DataFrame(data)
print(f" {len(data)} data scraping complete..from googleAPIs..fully complete records are saved in df_books_api")

[1/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Alexander the Great
[2/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Autobiography Of A Yogi
[3/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Be Your Own Sunshine by James allen
[4/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Black Holes: The Reith Lectures
[5/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Dark Horse ( Hindi Edition)
[6/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:David Copperfield : Indiana Illustrated Classics
[7/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:Deep and Silent Waters
[8/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:How To Be Rich
[9/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:How to Stop Worrying and Start Living
[10/3390] Fetching: https://www.googleapis.com/books/v1/volumes?q=intitle:How 

In [79]:
print("now checking the book data form goggleAPIs")
df_books_api.head()

now checking the book data form goggleAPIs


,Title,Authors,Publisher,Published Date,TextSnippet,ISBN_13,PageCount,Categories,Language,ListPrice,RetailPrice,Title_clean
0,Alexander the Great,Martin Howard,Bloomsbury Publishing,2012-05-24,"Alexander the Great, one of the most courageou...",9781408163702,129,Juvenile Nonfiction,en,583.00,291.50,alexander the great
1,Autobiography of a Yogi,"Paramahansa Yogananda, Sheba Blake",Sheba Blake Publishing,2022-01-10,The famous opera singer Amelita Galli-Curci sa...,9783986770105,645,Biography & Autobiography,en,319.00,223.30,autobiography of a yogi
2,Be Your Own Sunshine,"James Allen,",Sristhi Publishers & Distributors,2022-02-10,"... <b>own</b> pace, as per <b>our</b> comfort...",9789387022850,114,Self-Help,en,153.40,107.38,be your own sunshine
3,Black Holes: The Reith Lectures,Stephen Hawking,Random House,2016-05-05,Stephen Hawking. CLIFFHANGER In my previous <b...,9781473541986,82,Science,en,87.61,87.61,black holes: the reith lectures
4,Deep and Silent Waters,Charlotte Lamb,Hachette UK,2013-03-28,... <b>silence</b> all evening in a little bar...,9781444770322,353,Fiction,en,350.46,245.32,deep and silent waters


In [73]:
#now i need to merge records from df_books_api and the df dataframe to make one file that contain google apis data and adore also
#but for understanding i give respective name during cleaning process
# Normalize titles
df["Title_clean"] = df["Title_clean"].str.strip().str.lower()
df_books_api["Title_clean"] = df_books_api["Title_clean"].str.strip().str.lower()

# Keep only the first occurrence of each Title_clean in scrape
df_unique = df.drop_duplicates(subset="Title_clean")

# Merge: keep all API rows, add scrape data if Title_clean matches
final_extraction = pd.merge(
    df_books_api,
    df_unique[["Title_clean", "Current Price", "Original Price", "Rating"]],
    on="Title_clean",
    how="left"
)

print("Rows in df_books_api:", len(df_books_api))
print("Rows in final_extraction:", len(final_extraction))

Rows in df_books_api: 2462
Rows in final_extraction: 2462


In [77]:
final_extraction.head()
# Now Save to CSV
final_extraction.to_csv("final_extracted_data.csv", index=False, encoding="utf-8")
print("Saved final_extracted_data.csv with", len(final_extraction), "rows")

Saved final_extracted_data.csv with 2462 rows
